In [163]:
import pandas as pd

df = pd.read_excel("data/train.xlsx")

df

,review_description,rating
0,شركه زباله و سواقين بتبرشم و مفيش حتي رقم للشك...,-1
1,خدمة الدفع عن طريق الكي نت توقفت عندي اصبح فقط...,1
2,تطبيق غبي و جاري حذفه ، عاملين اكواد خصم و لما...,-1
3,فعلا تطبيق ممتاز بس لو فى امكانية يتيح لمستخدم...,1
4,سيء جدا ، اسعار رسوم التوصيل لا تمت للواقع ب ص...,-1
...,...,...
32031,التطبيق اصبح سيء للغايه نقوم بطلب لا يتم وصول ...,-1
32032,y love you,1
32033,الباقه بتخلص وبشحن مرتين باقه اضافيه ١٠٠ جنيه,-1
32034,تطبيق فاشل وصلني الطلب ناقص ومش ينفع اعمل حاجة...,-1


In [164]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [165]:
df = df.dropna()

df = df.drop_duplicates()

df["review_description"] = df["review_description"].astype(str)

In [166]:
arabic_stopwords = set(stopwords.words("arabic"))

negation_words = {
    "لا", "لم", "لن", "ليس", "ما", "غير",
    "بدون", "دون", "ولا", "ليسوا", "ليست"
}

arabic_stopwords = arabic_stopwords - negation_words

In [167]:
!pip install emoji


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
emoji_df = pd.read_csv("emoji.csv")
# أو
# emoji_df = pd.read_excel("emoji.xlsx")

In [ ]:
emoji_dict = dict(zip(
    emoji_df["emoji"],
    emoji_df["meaning"]
))

In [ ]:
def replace_emojis(text):
    for emj, meaning in emoji_dict.items():
        text = text.replace(emj, " " + meaning + " ")
    return text

In [ ]:
import re
import emoji
from nltk.tokenize import word_tokenize

# Function to preprocess Arabic text
def preprocess(text):

    # ==========================
    # Remove URLs
    # ==========================
    text = re.sub(r"http\S+|www\S+", "", text)

    # ==========================
    # Remove HTML Tags
    # ==========================
    text = re.sub(r"<.*?>", "", text)

    # ==========================
    # Remove Emojis
    # ==========================
    text = replace_emojis(text)
    # ==========================
    # Remove Mentions (@username)
    # and Hashtags (#topic)
    # ==========================
    text = re.sub(r"[@#]\w+", "", text)

    # ==========================
    # Remove English Letters
    # ==========================
    text = re.sub(r"[A-Za-z]+", "", text)

    # ==========================
    # Remove Numbers
    # (Arabic & English)
    # ==========================
    text = re.sub(r"[0-9٠-٩]", "", text)

    # ==========================
    # Remove Punctuation
    # ==========================
    text = re.sub(r"[^\w\s]", " ", text)

    # ==========================
    # Remove Arabic Diacritics
    # ==========================
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)

    # ==========================
    # Normalize Arabic Letters
    # ==========================
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)

    # ==========================
    # Remove Repeated Characters
    # Example:
    # راااائع -> رائع
    # حلوووو -> حلو
    # ==========================
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # ==========================
    # Keep Arabic Letters Only
    # ==========================
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # ==========================
    # Remove Extra Spaces
    # ==========================
    text = re.sub(r"\s+", " ", text).strip()

    # ==========================
    # Tokenization
    # ==========================
    words = word_tokenize(text)

    # ==========================
    # Remove Arabic Stopwords
    # ==========================
    words = [word for word in words if word not in arabic_stopwords]


    # ==========================
    # Join Tokens Again
    # ==========================
    return " ".join(words)

In [169]:
df["clean_text"] = df["review_description"].apply(preprocess)

df = df[df["clean_text"].str.strip() != ""]

df = df[df["clean_text"].str.split().str.len() >= 2]

df.reset_index(drop=True, inplace=True)

In [170]:

print(df["rating"].value_counts())

rating
 1    15091
-1    10355
 0     1360
Name: count, dtype: int64


In [171]:
df[["review_description", "clean_text", "rating"]].head(10)

,review_description,clean_text,rating
0,شركه زباله و سواقين بتبرشم و مفيش حتي رقم للشك...,شركه زباله سواقين بتبرشم مفيش حتي رقم للشكاوي ...,-1
1,خدمة الدفع عن طريق الكي نت توقفت عندي اصبح فقط...,خدمة الدفع طريق الكي نت توقفت عندي اصبح فقط ال...,1
2,تطبيق غبي و جاري حذفه ، عاملين اكواد خصم و لما...,تطبيق غبي جاري حذفه عاملين اكواد خصم نستخدمها ...,-1
3,فعلا تطبيق ممتاز بس لو فى امكانية يتيح لمستخدم...,فعلا تطبيق ممتاز امكانية يتيح لمستخدم التطبيق ...,1
4,سيء جدا ، اسعار رسوم التوصيل لا تمت للواقع ب ص...,سيء جدا اسعار رسوم التوصيل لا تمت للواقع صله,-1
5,قعد عشرين سنة يدور على سائق بس اما عن توصيل ال...,قعد سنة يدور علي سايق اما توصيل الاشياء جيد جدا,0
6,احلئ تطبيق,احلي تطبيق,1
7,رائع واو مدهش,رايع مدهش,1
8,مکو بس البحرین وعمان وغیرهه بس العراق مکو یعنی...,مکو البحرین وعمان وغیرهه العراق مکو یعنی نجمه ...,-1
9,تطبيق جميل يستاهل الخمس نجوم👍👍👍,تطبيق جميل يستاهل الخمس نجوم,1


In [172]:
lengths = df["clean_text"].str.split().apply(len)

print(lengths.describe())

count    26806.000000
mean         9.215922
std         10.854824
min          2.000000
25%          3.000000
50%          6.000000
75%         11.000000
max        321.000000
Name: clean_text, dtype: float64


In [173]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["rating"]

y = df["rating"].replace({
    -1: 0,
     0: 1,
     1: 2
})
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [174]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))

print(class_weights)

{np.int64(0): np.float64(0.8628681796233704), np.int64(1): np.float64(6.569852941176471), np.int64(2): np.float64(0.5921139827700463)}


In [175]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 30000
MAX_LEN = 50

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(x_train)

x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

x_train = pad_sequences(
    x_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

x_test = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [176]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

lstm = Sequential()

# Embedding Layer
lstm.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    )
)

# LSTM Layer
lstm.add(
    LSTM(
        64,
        dropout=0.3,
        recurrent_dropout=0.3
    )
)

# Hidden Layer
lstm.add(Dense(64, activation="relu"))

# Dropout Layer
lstm.add(Dropout(0.5))

# Output Layer
lstm.add(Dense(3, activation="softmax"))

optimizer = Adam(learning_rate=0.0005)

lstm.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = lstm.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weights
)

end_time = time.time()

training_time_lstm = end_time - start_time

print(f"Training Time: {training_time_lstm:.2f} seconds")
print(f"Training Time: {training_time_lstm/60:.2f} minutes")

c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 24s 74ms/step - accuracy: 0.3288 - loss: 1.1045 - val_accuracy: 0.5738 - val_loss: 1.0931
Epoch 2/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 22s 82ms/step - accuracy: 0.3195 - loss: 1.1026 - val_accuracy: 0.6454 - val_loss: 1.0338
Epoch 3/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 21s 79ms/step - accuracy: 0.6433 - loss: 1.0121 - val_accuracy: 0.7564 - val_loss: 0.9336
Epoch 4/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 23s 87ms/step - accuracy: 0.6043 - loss: 0.9905 - val_accuracy: 0.5796 - val_loss: 0.9900
Epoch 5/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 20s 75ms/step - accuracy: 0.6859 - loss: 0.9568 - val_accuracy: 0.7724 - val_loss: 0.8713
Epoch 6/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 38s 142ms/step - accuracy: 0.7319 - loss: 0.9337 - val_accuracy: 0.7314 - val_loss: 0.9067
Epoch 7/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 23s 86ms/step - accuracy: 0.7327 - loss: 0.9199 - val_accuracy: 0.7647 - val_loss: 0.8665
Epoch 8/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 25s 92ms/step - accuracy: 0.7519 - loss: 0.9097 -

In [185]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

gru = Sequential()

# Embedding Layer
gru.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    )
)

# GRU Layer
gru.add(
    GRU(
        64,
        dropout=0.3,
        recurrent_dropout=0.3
    )
)

# Hidden Layer
gru.add(Dense(64, activation="relu"))

# Dropout Layer
gru.add(Dropout(0.5))

# Output Layer
gru.add(Dense(3, activation="softmax"))

optimizer = Adam(learning_rate=0.0005)

gru.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = gru.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weights
)

end_time = time.time()

training_time_gru = end_time - start_time

print(f"Training Time: {training_time_gru:.2f} seconds")
print(f"Training Time: {training_time_gru/60:.2f} minutes")

Epoch 1/20


c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


269/269 ━━━━━━━━━━━━━━━━━━━━ 18s 56ms/step - accuracy: 0.2951 - loss: 1.1035 - val_accuracy: 0.3859 - val_loss: 1.0923
Epoch 2/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 16s 58ms/step - accuracy: 0.2244 - loss: 1.1034 - val_accuracy: 0.3859 - val_loss: 1.0612
Epoch 3/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 15s 55ms/step - accuracy: 0.3760 - loss: 1.1023 - val_accuracy: 0.0588 - val_loss: 1.1011
Epoch 4/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 15s 54ms/step - accuracy: 0.2318 - loss: 1.0970 - val_accuracy: 0.3859 - val_loss: 1.2164
Training Time: 63.04 seconds
Training Time: 1.05 minutes


In [178]:
lstm.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 50, 128)        │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,681,291 (44.56 MB)

 Trainable params: 3,893,763 (14.85 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,787,528 (29.71 MB)

In [179]:
gru.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ (None, 50, 128)        │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,644,811 (44.42 MB)

 Trainable params: 3,881,603 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,763,208 (29.61 MB)

In [180]:
loss_lstm, accuracy_lstm = lstm.evaluate(
    x_test,
    y_test
)
print("Loss:", loss_lstm)
print("Accuracy:", accuracy_lstm)

168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7811 - loss: 0.8405
Loss: 0.8405256271362305
Accuracy: 0.7810518741607666


In [181]:
loss_gru, accuracy_gru = gru.evaluate(x_test, y_test)

print("Loss:", loss_gru)
print("Accuracy:", accuracy_gru)

168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5712 - loss: 1.0813
Loss: 1.081346035003662
Accuracy: 0.571242094039917


In [182]:
from sklearn.metrics import classification_report
import numpy as np

y_pred_lstm = lstm.predict(x_test)

y_pred_lstm = np.argmax(y_pred_lstm, axis=1)

print("LSTM Classification Report")
print(
    classification_report(
        y_test,
        y_pred_lstm,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

168/168 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
LSTM Classification Report
              precision    recall  f1-score   support

    Negative       0.68      0.89      0.77      2071
     Neutral       0.00      0.00      0.00       272
    Positive       0.89      0.78      0.83      3019

    accuracy                           0.78      5362
   macro avg       0.52      0.56      0.53      5362
weighted avg       0.76      0.78      0.76      5362



c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [183]:
from sklearn.metrics import classification_report
import numpy as np

y_pred_gru = gru.predict(x_test)

y_pred_gru = np.argmax(y_pred_gru, axis=1)

print("GRU Classification Report")
print(
    classification_report(
        y_test,
        y_pred_gru,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

168/168 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step
GRU Classification Report
              precision    recall  f1-score   support

    Negative       0.75      0.03      0.06      2071
     Neutral       0.06      0.01      0.01       272
    Positive       0.57      0.99      0.72      3019

    accuracy                           0.57      5362
   macro avg       0.46      0.34      0.27      5362
weighted avg       0.61      0.57      0.43      5362



In [184]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# LSTM Metrics
lstm_accuracy = accuracy_score(y_test, y_pred_lstm)
lstm_precision = precision_score(y_test, y_pred_lstm, average="weighted")
lstm_recall = recall_score(y_test, y_pred_lstm, average="weighted")
lstm_f1 = f1_score(y_test, y_pred_lstm, average="weighted")

# GRU Metrics
gru_accuracy = accuracy_score(y_test, y_pred_gru)
gru_precision = precision_score(y_test, y_pred_gru, average="weighted")
gru_recall = recall_score(y_test, y_pred_gru, average="weighted")
gru_f1 = f1_score(y_test, y_pred_gru, average="weighted")

comparison = pd.DataFrame({
    "Model": ["LSTM", "GRU"],
    "Accuracy": [lstm_accuracy, accuracy_gru],
    "Loss": [loss_lstm, loss_gru],
    "Precision": [lstm_precision, gru_precision],
    "Recall": [lstm_recall, gru_recall],
    "F1-Score": [lstm_f1, gru_f1],
    "Training Time (sec)": [
        training_time_lstm,
        training_time_gru
    ]

})

comparison

c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Model,Accuracy,Loss,Precision,Recall,F1-Score,Training Time (sec)
0,LSTM,0.781052,0.840526,0.761140,0.781052,0.763483,236.541540
1,GRU,0.571242,1.081346,0.614425,0.571242,0.433399,165.574916
